# Modelo de recomendación mediante factores latentes

In [2]:
import pandas as pd
import numpy as np

## 1. Carga y preparación de los datos

In [3]:
ratings = pd.read_csv("../data/raw/ml-latest-small/ratings.csv")

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## 2. Preparación de usuarios y películas

Para aplicar la factorización matricial, los identificadores originales de usuarios y películas se transforman en índices consecutivos. Estos índices permitirán acceder directamente a las matrices de factores latentes que se crearán posteriormente.

In [4]:
usuarios = sorted(ratings["userId"].unique())
peliculas = sorted(ratings["movieId"].unique())

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(usuarios)
}

movie_to_idx = {
    movie_id: idx
    for idx, movie_id in enumerate(peliculas)
}

n_usuarios = len(usuarios)
n_peliculas = len(peliculas)

print("Número de usuarios:", n_usuarios)
print("Número de películas:", n_peliculas)

Número de usuarios: 610
Número de películas: 9724


## 3. Inicialización del modelo de factores latentes

Se define el número de factores latentes que representarán a cada usuario y película. A partir de ellos se crean las matrices de factores de usuarios y películas, junto con los sesgos asociados a cada uno.

Los parámetros se inicializan antes del entrenamiento y posteriormente serán ajustados a partir de las valoraciones disponibles.

In [5]:
n_factores = 20

np.random.seed(42)

P = np.random.normal(
    scale=0.1,
    size=(n_usuarios, n_factores)
)

Q = np.random.normal(
    scale=0.1,
    size=(n_peliculas, n_factores)
)

bias_usuarios = np.zeros(n_usuarios)
bias_peliculas = np.zeros(n_peliculas)

media_global = ratings["rating"].mean()

print("Forma de P:", P.shape)
print("Forma de Q:", Q.shape)
print("Media global:", media_global)

Forma de P: (610, 20)
Forma de Q: (9724, 20)
Media global: 3.501556983616962


## 4. Asociación de los identificadores con los factores latentes

Se incorporan al conjunto de valoraciones los índices internos de usuarios y películas creados anteriormente. 
Estos índices permitirán acceder durante el entrenamiento a los factores latentes correspondientes a cada valoración.

In [6]:
ratings_modelo = ratings.copy()

ratings_modelo["user_idx"] = ratings_modelo["userId"].map(user_to_idx)
ratings_modelo["movie_idx"] = ratings_modelo["movieId"].map(movie_to_idx)

ratings_modelo.head()

,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1,4.0,964982703,0,0
1,1,3,4.0,964981247,0,2
2,1,6,4.0,964982224,0,5
3,1,47,5.0,964983815,0,43
4,1,50,5.0,964982931,0,46


## 5. Entrenamiento mediante descenso de gradiente estocástico

El modelo se entrena recorriendo las valoraciones conocidas y comparando cada puntuación real con la predicción realizada mediante los factores latentes y los sesgos.

A partir del error obtenido se actualizan progresivamente los parámetros del modelo mediante descenso de gradiente estocástico (SGD).

In [7]:
learning_rate = 0.01
regularizacion = 0.02
n_epocas = 20

In [8]:
errores_epoca = []

for epoca in range(n_epocas):

    # Mezclamos las valoraciones en cada época
    datos_epoca = ratings_modelo.sample(frac=1, random_state=42 + epoca)

    error_cuadratico = 0

    for fila in datos_epoca.itertuples():

        u = fila.user_idx
        i = fila.movie_idx
        rating_real = fila.rating

        # Predicción actual
        prediccion = (
            media_global
            + bias_usuarios[u]
            + bias_peliculas[i]
            + np.dot(P[u], Q[i])
        )

        # Error de la predicción
        error = rating_real - prediccion

        error_cuadratico += error ** 2

        # Guardamos los factores actuales del usuario
        P_u_anterior = P[u].copy()

        # Actualización de los sesgos
        bias_usuarios[u] += learning_rate * (
            error - regularizacion * bias_usuarios[u]
        )

        bias_peliculas[i] += learning_rate * (
            error - regularizacion * bias_peliculas[i]
        )

        # Actualización de los factores latentes
        P[u] += learning_rate * (
            error * Q[i] - regularizacion * P[u]
        )

        Q[i] += learning_rate * (
            error * P_u_anterior - regularizacion * Q[i]
        )

    rmse_epoca = np.sqrt(error_cuadratico / len(ratings_modelo))
    errores_epoca.append(rmse_epoca)

    print(
        f"Época {epoca + 1}/{n_epocas} - "
        f"RMSE entrenamiento: {rmse_epoca:.4f}"
    )

Época 1/20 - RMSE entrenamiento: 0.9371
Época 2/20 - RMSE entrenamiento: 0.8856
Época 3/20 - RMSE entrenamiento: 0.8668
Época 4/20 - RMSE entrenamiento: 0.8541
Época 5/20 - RMSE entrenamiento: 0.8437
Época 6/20 - RMSE entrenamiento: 0.8343
Época 7/20 - RMSE entrenamiento: 0.8249
Época 8/20 - RMSE entrenamiento: 0.8148
Época 9/20 - RMSE entrenamiento: 0.8035
Época 10/20 - RMSE entrenamiento: 0.7904
Época 11/20 - RMSE entrenamiento: 0.7759
Época 12/20 - RMSE entrenamiento: 0.7597
Época 13/20 - RMSE entrenamiento: 0.7425
Época 14/20 - RMSE entrenamiento: 0.7250
Época 15/20 - RMSE entrenamiento: 0.7075
Época 16/20 - RMSE entrenamiento: 0.6906
Época 17/20 - RMSE entrenamiento: 0.6741
Época 18/20 - RMSE entrenamiento: 0.6587
Época 19/20 - RMSE entrenamiento: 0.6441
Época 20/20 - RMSE entrenamiento: 0.6307


## 6. Predicción de valoraciones

Una vez entrenado el modelo, se define una función que permite estimar la valoración que un usuario asignaría a una determinada película utilizando sus factores latentes y los sesgos aprendidos.

In [9]:
def predecir_rating(usuario_id, pelicula_id, limitar=True):

    if usuario_id not in user_to_idx:
        return np.nan

    if pelicula_id not in movie_to_idx:
        return np.nan

    u = user_to_idx[usuario_id]
    i = movie_to_idx[pelicula_id]

    prediccion = (
        media_global
        + bias_usuarios[u]
        + bias_peliculas[i]
        + np.dot(P[u], Q[i])
    )

    if limitar:
        prediccion = np.clip(prediccion, 0.5, 5.0)

    return prediccion

In [10]:
usuario_id = 1
pelicula_id = 1

prediccion = predecir_rating(usuario_id, pelicula_id)

print("Predicción:", prediccion)

Predicción: 4.577580077037967


In [11]:
ratings[
    (ratings["userId"] == usuario_id) &
    (ratings["movieId"] == pelicula_id)
]

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703


## 7. Generación de recomendaciones

A partir del modelo entrenado se genera una función capaz de recomendar películas a un usuario.

Para ello, se excluyen las películas que el usuario ya ha valorado, se predice su puntuación para el resto de películas y se seleccionan aquellas con mayor valoración estimada.

In [12]:
movies = pd.read_csv("../data/raw/ml-latest-small/movies.csv")

movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [13]:
def recomendar_peliculas(usuario_id, n=10):

    if usuario_id not in user_to_idx:
        return None

    # Películas ya valoradas por el usuario
    peliculas_vistas = set(
        ratings.loc[
            ratings["userId"] == usuario_id,
            "movieId"
        ]
    )

    recomendaciones = []

    # Predecimos únicamente películas no vistas
    for pelicula_id in peliculas:

        if pelicula_id not in peliculas_vistas:

            prediccion = predecir_rating(
                usuario_id,
                pelicula_id,
                limitar=False
)

            recomendaciones.append(
                (pelicula_id, prediccion)
            )

    # Ordenamos de mayor a menor predicción
    recomendaciones.sort(
        key=lambda x: x[1],
        reverse=True
    )

    # Seleccionamos las mejores
    top_n = recomendaciones[:n]

    resultado = pd.DataFrame(
        top_n,
        columns=["movieId", "prediccion"]
    )

    # Añadimos título y géneros
    resultado = resultado.merge(
        movies,
        on="movieId",
        how="left"
    )

    return resultado[
        ["movieId", "title", "genres", "prediccion"]
    ]

In [14]:
recomendar_peliculas(usuario_id=1, n=10)

,movieId,title,genres,prediccion
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,5.302589
1,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama,5.262667
2,1178,Paths of Glory (1957),Drama|War,5.246167
3,3451,Guess Who's Coming to Dinner (1967),Drama,5.218589
4,720,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,5.213421
5,1104,"Streetcar Named Desire, A (1951)",Drama,5.203551
6,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,5.202758
7,951,His Girl Friday (1940),Comedy|Romance,5.188621
8,1248,Touch of Evil (1958),Crime|Film-Noir|Thriller,5.187683
9,1272,Patton (1970),Drama|War,5.177451


## 8. Comprobación de las recomendaciones

Para comprobar de forma cualitativa el comportamiento del recomendador, se observan algunas de las películas mejor valoradas por un usuario y se comparan con las recomendaciones generadas por el modelo.

In [15]:
valoraciones_usuario = (
    ratings[ratings["userId"] == 1]
    .merge(movies, on="movieId")
    .sort_values("rating", ascending=False)
)

valoraciones_usuario[
    ["title", "genres", "rating"]
].head(15)

,title,genres,rating
3,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,5.0
4,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,5.0
6,Bottle Rocket (1996),Adventure|Comedy|Crime|Romance,5.0
13,Dumb & Dumber (Dumb and Dumber) (1994),Adventure|Comedy,5.0
11,Billy Madison (1995),Comedy,5.0
10,Desperado (1995),Action|Romance|Western,5.0
9,Canadian Bacon (1995),Comedy|War,5.0
8,Rob Roy (1995),Action|Drama|Romance|War,5.0
35,Pinocchio (1940),Animation|Children|Fantasy|Musical,5.0
31,Tombstone (1993),Action|Drama|Western,5.0


In [16]:
recomendar_peliculas(usuario_id=1, n=10)

,movieId,title,genres,prediccion
0,318,"Shawshank Redemption, The (1994)",Crime|Drama,5.302589
1,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama,5.262667
2,1178,Paths of Glory (1957),Drama|War,5.246167
3,3451,Guess Who's Coming to Dinner (1967),Drama,5.218589
4,720,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,5.213421
5,1104,"Streetcar Named Desire, A (1951)",Drama,5.203551
6,4973,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,5.202758
7,951,His Girl Friday (1940),Comedy|Romance,5.188621
8,1248,Touch of Evil (1958),Crime|Film-Noir|Thriller,5.187683
9,1272,Patton (1970),Drama|War,5.177451


In [17]:
print("P:", P.shape)
print("Q:", Q.shape)
print("bias usuarios:", len(bias_usuarios))
print("bias películas:", len(bias_peliculas))
print("usuarios mapping:", len(user_to_idx))
print("películas mapping:", len(movie_to_idx))
print("media global:", media_global)

P: (610, 20)
Q: (9724, 20)
bias usuarios: 610
bias películas: 9724
usuarios mapping: 610
películas mapping: 9724
media global: 3.501556983616962


In [18]:
from pathlib import Path

ruta_modelo = Path("../models/factores_latentes.pkl")

ruta_modelo.parent.mkdir(
    parents=True,
    exist_ok=True
)

print(ruta_modelo)

..\models\factores_latentes.pkl


In [19]:
modelo_factores_latentes = {
    "P": P,
    "Q": Q,
    "bias_usuarios": bias_usuarios,
    "bias_peliculas": bias_peliculas,
    "media_global": media_global,
    "user_to_idx": user_to_idx,
    "movie_to_idx": movie_to_idx
}

print(modelo_factores_latentes.keys())

dict_keys(['P', 'Q', 'bias_usuarios', 'bias_peliculas', 'media_global', 'user_to_idx', 'movie_to_idx'])


In [20]:
import pickle

with open(ruta_modelo, "wb") as archivo:
    pickle.dump(
        modelo_factores_latentes,
        archivo
    )

print("Modelo guardado correctamente.")

Modelo guardado correctamente.


In [21]:
with open(ruta_modelo, "rb") as archivo:
    modelo_cargado = pickle.load(archivo)

print(modelo_cargado.keys())
print("P:", modelo_cargado["P"].shape)
print("Q:", modelo_cargado["Q"].shape)

dict_keys(['P', 'Q', 'bias_usuarios', 'bias_peliculas', 'media_global', 'user_to_idx', 'movie_to_idx'])
P: (610, 20)
Q: (9724, 20)
